In [23]:
import pathlib
import sys
import duckdb
import pandas as pd

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())

TICKER = 'ARIS'
print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [24]:
with duckdb.connect(DB, read_only=True) as con:
    income = con.execute(
        'SELECT variant, period, "Fiscal Year", "Fiscal Period", "Publish Date", "Report Date", "Revenue", "Net Income (Common)"'
        ' FROM income WHERE "Ticker" = ? ORDER BY period',
        [TICKER]
    ).df()

print(f'{len(income)} income rows for {TICKER}')
income

20 income rows for ARIS


,variant,period,Fiscal Year,Fiscal Period,Publish Date,Report Date,Revenue,Net Income (Common)
0,A,2020A,2020,FY,2021-03-01,2020-12-31,171472000.0,-4328000
1,A,2021A,2021,FY,2022-03-01,2021-12-31,229251000.0,1112000
2,Q,2021Q3,2021,Q3,2021-11-10,2021-09-30,59499000.0,-20743000
3,Q,2021Q4,2021,Q4,2022-03-01,2021-12-31,66979000.0,14458000
4,A,2022A,2022,FY,2023-03-09,2022-12-31,321001000.0,1700000
5,Q,2022Q1,2022,Q1,2022-05-10,2022-03-31,70969000.0,-2222000
6,Q,2022Q2,2022,Q2,2022-08-04,2022-06-30,76386000.0,1394000
7,Q,2022Q3,2022,Q3,2022-11-10,2022-09-30,90776000.0,699000
8,Q,2022Q4,2022,Q4,2023-03-09,2022-12-31,82870000.0,1829000
9,A,2023A,2023,FY,2024-02-29,2023-12-31,392118000.0,18888000


In [25]:
with duckdb.connect(DB, read_only=True) as con:
    filings = con.execute(
        'SELECT period, url, error FROM sec_filings WHERE ticker = ? ORDER BY period',
        [TICKER]
    ).df()

print(f'{len(filings)} sec_filings rows for {TICKER}')
filings

20 sec_filings rows for ARIS


,period,url,error
0,2020A,NaN,no matching filing
1,2021A,NaN,no matching filing
2,2021Q3,https://www.sec.gov/Archives/edgar/data/196450...,NaN
3,2021Q4,NaN,no matching filing
4,2022A,NaN,no matching filing
5,2022Q1,https://www.sec.gov/Archives/edgar/data/196450...,NaN
6,2022Q2,https://www.sec.gov/Archives/edgar/data/196450...,NaN
7,2022Q3,https://www.sec.gov/Archives/edgar/data/196450...,NaN
8,2022Q4,NaN,no matching filing
9,2023A,NaN,no matching filing


In [26]:
# Join income periods with filings to check coverage
merged = (
    income[['variant', 'period', 'Publish Date']]
    .merge(filings, on='period', how='left')
)
merged

,variant,period,Publish Date,url,error
0,A,2020A,2021-03-01,NaN,no matching filing
1,A,2021A,2022-03-01,NaN,no matching filing
2,Q,2021Q3,2021-11-10,https://www.sec.gov/Archives/edgar/data/196450...,NaN
3,Q,2021Q4,2022-03-01,NaN,no matching filing
4,A,2022A,2023-03-09,NaN,no matching filing
5,Q,2022Q1,2022-05-10,https://www.sec.gov/Archives/edgar/data/196450...,NaN
6,Q,2022Q2,2022-08-04,https://www.sec.gov/Archives/edgar/data/196450...,NaN
7,Q,2022Q3,2022-11-10,https://www.sec.gov/Archives/edgar/data/196450...,NaN
8,Q,2022Q4,2023-03-09,NaN,no matching filing
9,A,2023A,2024-02-29,NaN,no matching filing


In [27]:
import requests
from irp.sources.sec_edgar import _load_ticker_map

ticker_map = _load_ticker_map()
cik_int = ticker_map[TICKER.upper()]
CIK = f"{cik_int:010d}"
print(f"{TICKER} → CIK {CIK}")

resp = requests.get(
    f"https://data.sec.gov/submissions/CIK{CIK}.json",
    headers={"User-Agent": "marc Duon marcdumon@msn.com", "Accept-Encoding": "gzip, deflate"},
)
resp.raise_for_status()
submissions = resp.json()["filings"]["recent"]

list(submissions.keys())

ARIS → CIK 0001964504


['accessionNumber',
 'filingDate',
 'reportDate',
 'acceptanceDateTime',
 'act',
 'form',
 'fileNumber',
 'filmNumber',
 'items',
 'core_type',
 'size',
 'isXBRL',
 'isInlineXBRL',
 'isXBRLNumeric',
 'primaryDocument',
 'primaryDocDescription']

In [31]:
def indexes_by_date(date: str) -> list[int]:
    """Return all indexes where filingDate == date (YYYY-MM-DD)."""
    return [i for i, d in enumerate(submissions["filingDate"]) if d == date]


def filing_url(index: int) -> str:
    """Return the SEC filing URL for the accessionNumber at the given index."""
    acc = submissions["accessionNumber"][index].replace("-", "")
    doc = submissions["primaryDocument"][index]
    cik_int = int(CIK)
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc}/{doc}"


# --- try it ---
# All unique filing dates (first 20)
sorted(set(submissions["filingDate"]))

['2023-09-06',
 '2023-09-07',
 '2023-09-08',
 '2023-09-13',
 '2023-09-14',
 '2023-10-06',
 '2023-10-10',
 '2023-10-31',
 '2023-11-03',
 '2023-11-09',
 '2023-11-27',
 '2023-12-06',
 '2024-01-02',
 '2024-01-16',
 '2024-01-30',
 '2024-02-01',
 '2024-02-15',
 '2024-02-22',
 '2024-03-07',
 '2024-04-08',
 '2024-04-15',
 '2024-05-14',
 '2024-05-15',
 '2024-05-16',
 '2024-05-23',
 '2024-06-28',
 '2024-07-03',
 '2024-07-09',
 '2024-07-16',
 '2024-08-13',
 '2024-08-23',
 '2024-09-13',
 '2024-09-25',
 '2024-10-08',
 '2024-10-21',
 '2024-10-24',
 '2024-10-31',
 '2024-11-04',
 '2024-11-05',
 '2024-11-07',
 '2024-11-12',
 '2024-11-13',
 '2024-11-14',
 '2024-11-29',
 '2025-01-15',
 '2025-02-20',
 '2025-03-06',
 '2025-03-13',
 '2025-04-08',
 '2025-04-15',
 '2025-04-24',
 '2025-04-28',
 '2025-05-08',
 '2025-05-12',
 '2025-05-13',
 '2025-05-15',
 '2025-06-30',
 '2025-07-07',
 '2025-07-08',
 '2025-07-09',
 '2025-07-17',
 '2025-07-24',
 '2025-07-25',
 '2025-08-07',
 '2025-08-08',
 '2025-08-12',
 '2025-08-

In [35]:
# Example: look up a Publish Date from the income table
date = str(income.iloc[0]["Publish Date"])[:10]
date = '2025-02-20'
print(f"Looking up date: {date}")

idxs = indexes_by_date(date)
print(f"Indexes: {idxs}")

for i in idxs:
    print(f"  [{i}] form={submissions['form'][i]}  reportDate={submissions['reportDate'][i]}  url={filing_url(i)}")

Looking up date: 2025-02-20
Indexes: [64]
  [64] form=6-K  reportDate=2025-02-20  url=https://www.sec.gov/Archives/edgar/data/1964504/000127956925000172/form6k.htm


In [9]:
with duckdb.connect(DB, read_only=True) as con:
    errors_df = con.execute(
        "SELECT ticker, period, error FROM sec_filings WHERE error IS NOT NULL ORDER BY error, ticker, period"
    ).df()

print(f"{len(errors_df)} error rows total")
errors_df.groupby("error").size().sort_values(ascending=False)

163 error rows total


error
ticker not found      104
no matching filing     59
dtype: int64

In [10]:
# Spot-check: pick a ticker from "no matching filing" errors and test the fix
from irp.sources.sec_edgar import sec_filing_url

sample = errors_df[errors_df["error"] == "no matching filing"].head(20)

for _, row in sample.iterrows():
    # get publish_date from income
    pub = income[income["period"] == row["period"]]["Publish Date"]
    publish_date = str(pub.iloc[0])[:10] if not pub.empty else None
    try:
        url = sec_filing_url(row["ticker"], row["period"], publish_date)
        status = f"OK → {url[:80]}"
    except Exception as e:
        status = f"FAIL: {e}"
    print(f"{row['ticker']:8s} {row['period']:8s} {publish_date}  {status}")

AA       2021Q1   2021-08-09  OK → https://www.sec.gov/Archives/edgar/data/1675149/000156459021039147/aa-10q_202106
AA       2022Q1   2022-08-09  OK → https://www.sec.gov/Archives/edgar/data/1675149/000156459022026349/aa-10q_202206
AA       2023Q1   2023-08-09  OK → https://www.sec.gov/Archives/edgar/data/1675149/000095017023035168/aa-20230630.h
AA       2024Q1   2024-08-09  OK → https://www.sec.gov/Archives/edgar/data/1675149/000095017024090092/aa-20240630.h
AA       2025Q1   None  FAIL: no matching filing
AACG     2021A    2022-05-31  FAIL: no matching filing
AACG     2022A    2023-05-22  FAIL: no matching filing
AACI     2021A    2022-05-31  OK → https://www.sec.gov/Archives/edgar/data/2092897/000119312526117694/d107030d10k.h
AACI     2023Q3   2024-02-09  FAIL: no matching filing
AAL      2021Q1   2021-08-09  OK → https://www.sec.gov/Archives/edgar/data/6201/000000620121000080/aal-20210630.htm
AAL      2022Q1   2022-08-09  OK → https://www.sec.gov/Archives/edgar/data/6201/0000006201